In [23]:
import pymongo
import pandas as pd
import os
import glob
import json

In [24]:
# MongoDB connection is unviable due to authorisation issues, DB cloned to system instead.
file_path = ".\\Shiksha_Copilot_DB"
collections = glob.glob(os.path.join(file_path, "*.json"))

print(collections)

['.\\Shiksha_Copilot_DB\\activityratingaggregates.json', '.\\Shiksha_Copilot_DB\\adminusers.json', '.\\Shiksha_Copilot_DB\\auditlogs.json', '.\\Shiksha_Copilot_DB\\baselinesurveys.json', '.\\Shiksha_Copilot_DB\\boards.json', '.\\Shiksha_Copilot_DB\\chapters.json', '.\\Shiksha_Copilot_DB\\chats.json', '.\\Shiksha_Copilot_DB\\classes.json', '.\\Shiksha_Copilot_DB\\facilities.json', '.\\Shiksha_Copilot_DB\\helpvideos.json', '.\\Shiksha_Copilot_DB\\lba_questions.json', '.\\Shiksha_Copilot_DB\\lessonchats.json', '.\\Shiksha_Copilot_DB\\lessonfeedbacks.json', '.\\Shiksha_Copilot_DB\\lessonplantemplates.json', '.\\Shiksha_Copilot_DB\\masterclasses.json', '.\\Shiksha_Copilot_DB\\masterlessons.json', '.\\Shiksha_Copilot_DB\\masterresources.json', '.\\Shiksha_Copilot_DB\\mastersubjects.json', '.\\Shiksha_Copilot_DB\\messages.json', '.\\Shiksha_Copilot_DB\\questionbankcaches.json', '.\\Shiksha_Copilot_DB\\questionbankcachesummaries.json', '.\\Shiksha_Copilot_DB\\questionbankconfigurations.json', 

In [25]:
db_info = []

for coll in collections:
    coll_path = os.path.basename(coll)
    size = os.path.getsize(coll) / (2^20) #size in MB
    file = open(coll, "r", encoding='utf-8', errors='ignore')
    schema = file.readline()
    
    if schema:
        sample_doc = json.loads(schema)
        fields = list(sample_doc.keys())
        file.seek(0)
        count = sum(1 for line in file) #number of records per collection
        db_info.append({
            "Collection": coll_path.replace(".json", ""),
            "Records": count,
            "Size_MB": round(size, 2),
            "Size_perRecord": round(size / count, 2),
            "Fields": fields
        })

db_df = pd.DataFrame(db_info)

In [26]:
print(db_df)

                    Collection  Records      Size_MB  Size_perRecord  \
0                   adminusers       24       428.00           17.83   
1                    auditlogs      575     18097.86           31.47   
2              baselinesurveys     7630    184923.14           24.24   
3                       boards       52       558.45           10.74   
4                     chapters      999     97212.18           97.31   
5                        chats      545      6411.45           11.76   
6                      classes    11780    545369.00           46.30   
7                   facilities        2        32.91           16.45   
8                lba_questions     7494    227392.27           30.34   
9                  lessonchats     1380    163924.27          118.79   
10             lessonfeedbacks    35657   1270757.95           35.64   
11         lessonplantemplates       14      4482.95          320.21   
12               masterclasses       20       181.91            

In [27]:
for _, row in db_df.iterrows():
    collection_name = row["Collection"]
    field_names = row["Fields"]

    possible_reference_fields = [
        field for field in field_names
        if "Id" in field and field != "_id"
    ]

    if possible_reference_fields:
        print(
            f"'{collection_name}' likely links to other collections "
            f"via: {', '.join(possible_reference_fields)}"
        )


'auditlogs' likely links to other collections via: userId
'baselinesurveys' likely links to other collections via: userId
'chapters' likely links to other collections via: subjectId
'chats' likely links to other collections via: userId
'classes' likely links to other collections via: schoolId
'lba_questions' likely links to other collections via: chapterId, correctOrderById
'lessonchats' likely links to other collections via: teacherId, recordId
'lessonfeedbacks' likely links to other collections via: teacherId, lessonId
'lessonplantemplates' likely links to other collections via: workFlowId
'masterlessons' likely links to other collections via: chapterId, templateId
'masterresources' likely links to other collections via: lessonId, chapterId, templateId
'messages' likely links to other collections via: chatHistoryId
'questionbankcaches' likely links to other collections via: chapterId
'questionbankcachesummaries' likely links to other collections via: questionBankConfigId
'questionban